# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamal1osama/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane (or freestyle) and why

**Selected Lane:** **Lane 2: Refresh / Content Opportunity Scoring**

**Why this lane?**
In organic search management, the single most constrained operational bottleneck is human editorial bandwidth. A large website or agency manages thousands of published URLs, but an editorial team can realistically audit, rewrite, or expand only 10 to 20 articles per sprint. Relying on arbitrary recency heuristics (such as "refresh anything older than 6 months") causes teams to waste scarce writing hours on content with zero upside while high-value, page-one articles quietly slip in search visibility.

Lane 2 directly addresses this resource-allocation bottleneck: scoring and ranking the content catalog to produce an evidence-backed review queue equipped with diagnostic reason codes (such as decay momentum, search demand volume, intent mismatch, and page-one decay risk). Rather than automating writing or making blind algorithmic bets, Lane 2 empowers human editors to focus their limited capacity where the expected value of intervention and traffic preservation is highest.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Robust data path resolution supporting local execution, repo root, and Google Colab
data_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Catalog Overview:")
print(f"- Total content items: {len(df):,}")
print(f"- Distinct client portfolios: {df['client_id'].nunique()}")
print(f"- Feature dimensions available: {df.shape[1]} columns")
print(f"- Active downward trend count: {(df['trend_direction'] == 'down').sum():,} ({(df['trend_direction'] == 'down').mean():.1%})")



Catalog Overview:
- Total content items: 30,000
- Distinct client portfolios: 32
- Feature dimensions available: 44 columns
- Active downward trend count: 16,262 (54.2%)


## 2. The question: decision, action, cost of a wrong call

### The Four Core Framing Questions

1. **What decision does this work improve?**
   It improves the sprint-level **editorial prioritization decision**: out of thousands of published pages in a client's library, which 10–20 pages should the content and SEO teams pull into the upcoming sprint for manual review, factual update, structural expansion, or metadata optimization?

2. **Who acts on the output, and what do they do?**
   - **The Actor:** Content marketing leads, SEO strategists, and subject-matter editors.
   - **The Action:** The editor opens the weekly ranked opportunity queue, reviews the top-priority candidates alongside their diagnostic reason codes (e.g., `declining_with_demand`, `page_one_decay_risk`, `low_ctr_visible_page`), verifies current SERP competitor landscape, and executes targeted content enhancements (updating outdated statistics, expanding under-developed subtopics, re-optimizing title tags, or aligning headings with emerging search queries).

3. **What does a wrong recommendation cost? (Asymmetric error costs)**
   - **Cost of a False Positive (Recommending a page that does not need refresh or has negligible upside):**
     Directly burns 4 to 8 hours of senior editorial and subject-matter expert labor rewriting content that either already met user intent or had negligible search demand. Because editorial capacity is strictly capped, every wasted hour represents an opportunity cost of an urgent page left decaying.
   - **Cost of a False Negative (Failing to flag a high-value page in active decay):**
     The page quietly slips from Page 1 (positions 1–5) down to Page 2 or lower. Since more than 90% of search clicks occur on Page 1, slipping off Page 1 results in immediate, sustained losses of qualified traffic, leads, and revenue. Re-establishing lost rank against entrenched competitors requires substantially more time and link equity than proactively defending an existing high-ranking position.

4. **Why does data or machine learning help at all?**
   Content decay is a noisy, multivariate phenomenon that cannot be reliably captured by simple one-factor rules. In the starter dataset:
   - Naive recency rules fail completely: the median `days_since_last_update` is identical (20.0 days) for both declining and non-declining articles.
   - Simple heuristic rules achieve a meager **24.0% Precision@50** (over 75% of flagged pages are false positives).
   - A learned model (such as Random Forest) combines non-linear interactions across search exposure, position decay trajectories, engagement rates, and content age, achieving **74.0% Precision@50** on held-out clients—tripling editorial efficiency.

---

### The One-Paragraph Problem Frame
> **For SEO content managers and editorial teams, deciding which 10–20 URLs from the content inventory to refresh during the upcoming sprint, we will build a ranked opportunity scoring model from observable search performance, engagement signals, and content metadata, scoring candidate pages by their likelihood of unaddressed traffic decay and opportunity measured by Precision@K (specifically Precision@20 and Precision@50) and Average Precision on client-holdout splits. A wrong call costs 4–8 hours of wasted high-cost editorial rewrite time, while a missed call allows critical Page 1 rankings to slip permanently to competitors. A plain rule isn't enough because content decay stems from tangled, non-linear interactions across visibility, intent, and engagement that simple recency thresholds cannot separate. We will claim only observed, directional, decision-support ranking improvements—never algorithmic guarantees or causal recovery claims.**

---

### Problem Specification
- **Task Type:** Ranking / Prioritization Scoring (backed by supervised classification probabilities for decay risk).
- **Unit of Analysis:** One pseudonymized content item (`content_id`) for a specific client (`client_id`) evaluated over an observation window.
- **Target:** Observed outcome of traffic decline / decay (in starter: `trend_direction == 'down'`; transitioning in the warehouse release to forward-window traffic loss over the subsequent 30 days).
- **Primary Metric:** Precision@K (K=20, K=50) and ROC-AUC / Average Precision evaluated on client-grouped holdout splits.


In [2]:
# Comparing Baseline Rules vs. Learned Models on Client-Holdout Splits (from starter pipeline outputs)
comparison_data = [
    {"Approach": "Baseline Rules (Heuristic)", "ROC-AUC": 0.627, "Avg Precision": 0.468, "Precision@50": 0.240, "Correct in Top 50": "12 / 50 (24%)"},
    {"Approach": "Logistic Regression", "ROC-AUC": 0.700, "Avg Precision": 0.522, "Precision@50": 0.400, "Correct in Top 50": "20 / 50 (40%)"},
    {"Approach": "Decision Tree", "ROC-AUC": 0.742, "Avg Precision": 0.575, "Precision@50": 0.540, "Correct in Top 50": "27 / 50 (54%)"},
    {"Approach": "Random Forest (Best Model)", "ROC-AUC": 0.750, "Avg Precision": 0.618, "Precision@50": 0.740, "Correct in Top 50": "37 / 50 (74%)"},
]
comparison_df = pd.DataFrame(comparison_data)
print("=== Baseline vs. Learned Model Performance (Client Holdout) ===")
print(comparison_df.to_string(index=False))
print("\nTakeaway: The learned Random Forest model triples the precision of the top 50 review queue compared to heuristic rules (74% vs 24%).")



=== Baseline vs. Learned Model Performance (Client Holdout) ===
                  Approach  ROC-AUC  Avg Precision  Precision@50 Correct in Top 50
Baseline Rules (Heuristic)    0.627          0.468          0.24     12 / 50 (24%)
       Logistic Regression    0.700          0.522          0.40     20 / 50 (40%)
             Decision Tree    0.742          0.575          0.54     27 / 50 (54%)
Random Forest (Best Model)    0.750          0.618          0.74     37 / 50 (74%)

Takeaway: The learned Random Forest model triples the precision of the top 50 review queue compared to heuristic rules (74% vs 24%).


## 3. Quick look at the data (2-3 real numbers)

To verify that Lane 2 is backed by genuine search dynamics and worth the next 7 weeks of investigation, we inspect the starter dataset (`data/raw/content_refresh_anonymized.csv`) and identify four foundational numbers:

1. **Massive Library Decay (16,262 items / 54.2%):**
   Across the 30,000 catalog items, over half (54.2%) have `trend_direction == 'down'`. Because no human team could ever manually audit 16,000 decaying URLs, a probabilistic scoring mechanism is essential to identify the highest-priority subset.

2. **Severe Demand Concentration (Top 10% capture 70.2% of impressions):**
   Search exposure follows an extreme Pareto distribution: the top 10% of pages by impression volume (pages with $\ge 9,207$ impressions) account for **70.2%** of all 90-day search impressions (109.5M out of 156.0M total). Refreshing a zero-demand tail page produces negligible business impact; prioritizing the right head and torso pages is where ranking defense matters.

3. **High-Stakes Page-One Decay (3,666 mature Page 1 assets declining):**
   There are **7,076** mature pages ($\ge 180$ days old) maintaining Page 1 rankings (`avg_position` between 1.0 and 10.0). Of these prime traffic-driving assets, **3,666 pages (51.8%)** are actively trending downward. These represent immediate decay risks that, if left unattended, will slip off Page 1.

4. **Failure of Naive Recency Heuristics (Identical median age of 20 days):**
   The median `days_since_last_update` is identical at **20.0 days** for both declining and non-declining content. This proves empirically that simple time-based heuristics cannot distinguish decay, providing clear justification for a multi-signal machine learning model.


In [3]:
# Compute and verify the supporting numbers from the starter dataset
total_items = len(df)
declining_items = int((df["trend_direction"] == "down").sum())
declining_pct = declining_items / total_items

# 1. Traffic Skew / Concentration
total_impressions = float(df["impressions_90d"].sum())
p90_threshold = float(df["impressions_90d"].quantile(0.90))
top_10pct_impressions = float(df[df["impressions_90d"] >= p90_threshold]["impressions_90d"].sum())
top_10pct_share = top_10pct_impressions / total_impressions

# 2. Page 1 Decay Risk (mature pages with average position between 1.0 and 10.0)
page_one_mature = df[(df["avg_position"] > 0) & (df["avg_position"] <= 10.0) & (df["content_age_days"] >= 180)]
page_one_declining = int((page_one_mature["trend_direction"] == "down").sum())
page_one_declining_pct = page_one_declining / len(page_one_mature)

# 3. Median Recency Comparison
median_recency_declining = float(df[df["trend_direction"] == "down"]["days_since_last_update"].median())
median_recency_non_declining = float(df[df["trend_direction"] != "down"]["days_since_last_update"].median())

print("=== Supporting Empirical Numbers from Starter Dataset ===")
print(f"1. Scale of Decay: {declining_items:,} / {total_items:,} items are declining ({declining_pct:.1%}).")
print(f"2. Demand Skew: Top 10% pages (>= {p90_threshold:,.0f} imp) capture {top_10pct_share:.1%} of total impressions ({top_10pct_impressions:,.0f} / {total_impressions:,.0f}).")
print(f"3. Page 1 Decay Risk: {page_one_declining:,} out of {len(page_one_mature):,} mature Page 1 assets are declining ({page_one_declining_pct:.1%}).")
print(f"4. Heuristic Failure: Median days since last update is {median_recency_declining:.1f} days for declining vs {median_recency_non_declining:.1f} days for non-declining content.")



=== Supporting Empirical Numbers from Starter Dataset ===
1. Scale of Decay: 16,262 / 30,000 items are declining (54.2%).
2. Demand Skew: Top 10% pages (>= 12,136 imp) capture 70.2% of total impressions (109,520,119 / 156,010,989).
3. Page 1 Decay Risk: 3,666 out of 7,076 mature Page 1 assets are declining (51.8%).
4. Heuristic Failure: Median days since last update is 20.0 days for declining vs 20.0 days for non-declining content.


## 4. Careful words: what I can and can't claim

Honest machine learning practice requires setting strict boundaries on what our results mean and how they can be interpreted.

### What This Work CAN Claim
- **Observational and Directional Signal:** We can report empirical correlations between historical search metrics (impressions, positions, CTR, engagement retention) and observed traffic trends.
- **Decision-Support Prioritization:** We can produce a ranked queue that orders content so that human editors review pages with high decay risk and high demand first, measurably improving review precision over random sampling or naive heuristics.
- **Empirical Model Validation:** We can validate that on client-holdout splits (unseen domains), a trained multivariate model achieves higher Precision@50 and ROC-AUC than hand-crafted baseline rules on historical data.

### What This Work CANNOT Claim
- **No Causal Recovery Claims:** We cannot claim that updating or rewriting a page *causes* search rankings or traffic to recover. Establishing causality requires randomized controlled experiments (A/B testing) or quasi-experimental causal inference, which observational search logs alone cannot establish.
- **No "Reverse-Engineering Google":** We do not claim to discover Google's proprietary search ranking algorithm or ranking weights. Search engines evaluate thousands of continuously evolving signals; our model is simply a practical predictor of traffic trajectory on client websites.
- **No Guarantee of Success:** A high opportunity score indicates that a page matches historical patterns of high-demand decay; it does not guarantee that editorial intervention will prevent traffic loss.
- **No Data Leakage:** We do not claim predictive validity unless features are strictly derived from prior observation windows with zero leakage from target outcomes or future calendar periods.


In [4]:
# Verification of Data Cleanliness and Privacy Safeguards
safe_columns = [
    col for col in df.columns 
    if not any(token in col.lower() for token in ["title", "url", "query", "domain", "client_name"])
]
print(f"Verified Safe Columns: {len(safe_columns)} / {len(df.columns)} columns contain observable or pseudonymized features.")
print("Zero raw PII, URLs, or client identities are stored or processed.")

# Sample observable feature distributions across trend directions
summary_by_trend = df.groupby("trend_direction")[["impressions_90d", "avg_position", "ctr"]].agg(["median", "mean"]).round(2)
print("\nObservable Metrics Summary by Trend Direction:")
print(summary_by_trend)



Verified Safe Columns: 44 / 44 columns contain observable or pseudonymized features.
Zero raw PII, URLs, or client identities are stored or processed.

Observable Metrics Summary by Trend Direction:
                impressions_90d          avg_position           ctr      
                         median     mean       median   mean median  mean
trend_direction                                                          
down                      961.0  4919.10         11.3  15.94   0.08  0.32
flat                        4.0    20.74          6.6  11.10   0.00  1.38
new                         3.0   164.13          0.0   9.91   0.00  1.30
stable                   1944.5  9213.62         11.4  16.33   0.17  0.52
up                        587.0  4716.08         15.3  22.51   0.09  0.57


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
